Cropping of Frames 
Done

In [2]:
import cv2
import os
from mtcnn import MTCNN
from tqdm import tqdm
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

def crop_faces_with_nested_structure(input_dir, output_dir):
    """Handle nested frame directory structure"""
    detector = MTCNN()
    
    for label in ['real', 'fake']:
        label_input_path = os.path.join(input_dir, label)
        label_output_path = os.path.join(output_dir, label)
        
        if not os.path.exists(label_input_path):
            print(f"❌ Directory not found: {label_input_path}")
            continue

        # Create output directory for this label
        os.makedirs(label_output_path, exist_ok=True)
        
        # Get all video folders (000, 001, etc.)
        video_folders = [d for d in os.listdir(label_input_path) 
                       if os.path.isdir(os.path.join(label_input_path, d))]
        
        if not video_folders:
            print(f"⚠️ No video folders found in {label_input_path}")
            continue

        print(f"\nProcessing {label} images...")
        
        total_processed = 0
        for video_folder in tqdm(video_folders, desc=label):
            video_folder_path = os.path.join(label_input_path, video_folder)
            
            # Get all frame images in this video folder
            frame_files = [f for f in os.listdir(video_folder_path) 
                         if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            
            for frame_file in frame_files:
                frame_path = os.path.join(video_folder_path, frame_file)
                output_filename = f"{video_folder}_{frame_file}"
                output_path = os.path.join(label_output_path, output_filename)
                
                try:
                    img = cv2.imread(frame_path)
                    if img is None:
                        continue

                    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    faces = detector.detect_faces(img_rgb)
                    if not faces:
                        continue

                    x, y, w, h = faces[0]['box']
                    x, y = max(0, x), max(0, y)
                    face = img_rgb[y:y+h, x:x+w]
                    face = cv2.resize(face, (224, 224))
                    
                    cv2.imwrite(output_path, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
                    total_processed += 1
                    
                except Exception as e:
                    continue

        print(f"Processed {total_processed} {label} images")

    print("\n✅ Face cropping completed!")

# Example usage
crop_faces_with_nested_structure("frames", "cropped_faces")

Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "C:\Users\bhara\AppData\Roaming\Python\Python313\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "C:\Users\bhara\AppData\Roaming\Python\Python313\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "C:\Users\bhara\AppData\Roaming\Python\Python313\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.



Processing real images...


real: 100%|██████████| 100/100 [42:47<00:00, 25.67s/it]


Processed 5992 real images

Processing fake images...


fake: 100%|██████████| 100/100 [29:27<00:00, 17.67s/it]

Processed 5996 fake images

✅ Face cropping completed!
